# Batch 7 — Post-Batch-6 follow-ups

Covers: **M1-13, M2-8, M2-30, M2-34, M2-54, M2-55, M2-56, M3-7**

A themed mix that doesn't overlap any existing notebook: config-value confirmation (M1-13), the dormant
"auto" health-information trigger mode (M2-8), a patient-payload mislabeling bug (M2-30), a shared
cwd-relative-storage-path bug (M2-34/M3-7), and the 3 retry-exhaustion siblings flagged right after Batch 6
(M2-54/M2-55/M2-56) — all now investigated and (where a real bug existed) fixed.

Every check below runs against the REAL repo code (`server/`, `tools/`), isolated via `harness.py`'s scratch
storage + `unittest.mock.patch` for outbound calls — nothing here touches the real `storage/`/`logs/`
directories or calls real ABDM.

In [1]:
import sys, os
from pathlib import Path

# Locate the repo root by walking up from wherever `jupyter execute` sets
# the working directory (its default is the notebook's OWN directory,
# tools/edge_case_testing/notebooks -- not the repo root) until we find a
# directory containing both server/ and tools/.
_here = Path.cwd()
REPO_ROOT = None
for _candidate in [_here, *_here.parents]:
    if (_candidate / "server").is_dir() and (_candidate / "tools").is_dir():
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root by walking up from cwd={_here}")

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "tools" / "edge_case_testing" / "notebooks"))
os.chdir(REPO_ROOT)  # matches a real `uvicorn` launch from the repo root -- also required so the M2-34/M3-7 cell below can correctly compare a deliberately-different subprocess cwd against the real repo root

import asyncio
import subprocess
from unittest.mock import patch

import harness
from harness import FakeResponse, CallRecorder, check

harness.activate_scratch_storage('batch7')


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_batch7_p2k5540j
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)


WindowsPath('C:/Users/hp/AppData/Local/Temp/edge_case_scratch_batch7_p2k5540j')

## M1-13 — MAX_OTP_ATTEMPTS can't accidentally become 0

**Confirm-only.** `MAX_OTP_ATTEMPTS` is a plain module-level `int` literal in `tools/m1_test_suite/login_runner.py`
— not read from any env var, config file, or user input at runtime — so there's no runtime code path that
could set it to 0 by accident. The test plan's own text already says this is "not really runnable right
now — skip unless that config value ever changes"; this cell just confirms the current value and that it's
a hardcoded constant, not a live config surface.

In [2]:
import tools.m1_test_suite.login_runner as login_runner

check("MAX_OTP_ATTEMPTS is currently 3, not 0", login_runner.MAX_OTP_ATTEMPTS == 3)
check("MAX_OTP_ATTEMPTS is a plain int (a hardcoded source constant, not something runtime-configurable)", isinstance(login_runner.MAX_OTP_ATTEMPTS, int))
print("[M1-13] No fix needed -- this can only ever change via a source-code edit, which is out of scope for a runtime edge case.")


PASS -- MAX_OTP_ATTEMPTS is currently 3, not 0
PASS -- MAX_OTP_ATTEMPTS is a plain int (a hardcoded source constant, not something runtime-configurable)
[M1-13] No fix needed -- this can only ever change via a source-code edit, which is out of scope for a runtime edge case.


## M2-8 — `HEALTH_INFORMATION_TRIGGER_MODE = "auto"` (dormant mode, never run before)

**Confirm-only — the code already handles this correctly.** `maybe_trigger_health_information_request()`
(`server/callbacks/services/health_information_trigger.py`) branches on the mode: `"manual"` (the real
default) is a no-op; `"auto"` calls `initiate_health_information_request()` with `hiu_id`/`hip_id`/
`date_range` pulled from the just-fetched consent detail, and cleanly logs+returns (no crash) if
`hiu_id`/`hip_id` are missing rather than trying to make the call anyway. Verified all three paths directly
against the real function.

In [3]:
import server.callbacks.services.health_information_trigger as trigger_module

# "auto" mode, complete consentDetail -> exactly one Block 2 pull, correct args passed through
recorder_auto = CallRecorder(FakeResponse(202))
with patch.object(trigger_module, "HEALTH_INFORMATION_TRIGGER_MODE", "auto"), \
     patch.object(trigger_module, "initiate_health_information_request", recorder_auto):
    trigger_module.maybe_trigger_health_information_request(
        "consent-m2-8-auto",
        {"hiu": {"id": "hiu-1"}, "hip": {"id": "hip-1"}, "permission": {"dateRange": {"from": "2026-01-01T00:00:00.000Z", "to": "2026-02-01T00:00:00.000Z"}}},
    )
check("'auto' mode with a complete consentDetail triggers exactly 1 Block 2 pull", recorder_auto.call_count == 1)
check("'auto' mode passes hiu_id/hip_id through correctly", recorder_auto.calls[0]["kwargs"].get("hiu_id") == "hiu-1" and recorder_auto.calls[0]["kwargs"].get("hip_id") == "hip-1")

# "auto" mode, missing hip.id -> clean no-op, no crash, no call attempted
recorder_missing = CallRecorder(FakeResponse(202))
with patch.object(trigger_module, "HEALTH_INFORMATION_TRIGGER_MODE", "auto"), \
     patch.object(trigger_module, "initiate_health_information_request", recorder_missing):
    try:
        trigger_module.maybe_trigger_health_information_request("consent-m2-8-missing", {"hiu": {"id": "hiu-1"}})
        check("'auto' mode with a missing hip.id does not crash", True)
    except Exception as exc:
        check(f"'auto' mode with a missing hip.id crashed with {type(exc).__name__} -- FAILED", False)
check("'auto' mode with a missing hip.id never attempts the Block 2 pull", recorder_missing.call_count == 0)

# default "manual" mode -> confirms the dormant mode really is off by default
recorder_manual = CallRecorder(FakeResponse(202))
with patch.object(trigger_module, "HEALTH_INFORMATION_TRIGGER_MODE", "manual"), \
     patch.object(trigger_module, "initiate_health_information_request", recorder_manual):
    trigger_module.maybe_trigger_health_information_request("consent-m2-8-manual", {"hiu": {"id": "hiu-1"}, "hip": {"id": "hip-1"}})
check("default 'manual' mode is a true no-op", recorder_manual.call_count == 0)
print("[M2-8] No fix needed -- 'auto' mode behaves exactly as documented, both on the happy path and on missing required fields.")


2026-08-15 20:30:06     [WAITING] HEALTH_INFORMATION_TRIGGER_MODE='auto' -- automatically requesting health information for this consent
2026-08-15 20:30:06     [ERROR] Cannot auto-trigger Health Information Request for consent consent-m2-8-missing -- missing hiu.id/hip.id in consentDetail.
2026-08-15 20:30:06  -> HEALTH_INFORMATION_TRIGGER_MODE='manual' -- Block 2 not auto-triggered; use the M3 CLI's Health Information Request flow when ready.


PASS -- 'auto' mode with a complete consentDetail triggers exactly 1 Block 2 pull
PASS -- 'auto' mode passes hiu_id/hip_id through correctly
PASS -- 'auto' mode with a missing hip.id does not crash
PASS -- 'auto' mode with a missing hip.id never attempts the Block 2 pull
PASS -- default 'manual' mode is a true no-op
[M2-8] No fix needed -- 'auto' mode behaves exactly as documented, both on the happy path and on missing required fields.


## M2-30 — patient with two different hospital record numbers at one facility mislabeled

**Real bug found and fixed.** `build_patient_payload()` (`server/callbacks/transformers/patient_transformer.py`)
grouped records by `hi_type` ALONE. If the same real-world patient has two different hospital record
numbers at the same facility (two distinct `patient_reference` values, e.g. from a later-reconciled
duplicate registration) and both records share an HI Type, the old code silently mislabeled the SECOND
record's care context under the FIRST record's `patient_reference`/`name` — `first_record` was always
`hi_type_records[0]`, picked once per `hi_type` group. Fixed by grouping by `(patient_reference, hi_type)`
instead, so each hospital record number keeps its own correct label. The common case (one patient_reference,
multiple care contexts of the same HI Type) is unaffected — verified as a regression check below.

In [4]:
from server.callbacks.transformers.patient_transformer import build_patient_payload

records = [
    {"patient_reference": "MR-AAA-001", "name": "Patient A record", "care_context_reference": "CC-1", "care_context_display": "OPConsultation 1", "hi_type": "OPConsultation"},
    {"patient_reference": "MR-AAA-002", "name": "Patient A record (2nd hospital record #)", "care_context_reference": "CC-2", "care_context_display": "OPConsultation 2", "hi_type": "OPConsultation"},
]
payload = build_patient_payload(records)

check("2 distinct patient_reference values sharing an HI Type -> 2 separate patient objects (not merged/mislabeled)", len(payload) == 2)
ref_numbers = {p["referenceNumber"] for p in payload}
check("each patient object keeps its OWN patient_reference as referenceNumber", ref_numbers == {"MR-AAA-001", "MR-AAA-002"})
check("each patient object has exactly its own 1 care context, not the other's", all(len(p["careContexts"]) == 1 for p in payload))

# Regression: same patient_reference + hi_type still collapses into ONE object, exactly as before this fix
records_same = [
    {"patient_reference": "MR-BBB-001", "name": "Patient B", "care_context_reference": "CC-3", "care_context_display": "d1", "hi_type": "Prescription"},
    {"patient_reference": "MR-BBB-001", "name": "Patient B", "care_context_reference": "CC-4", "care_context_display": "d2", "hi_type": "Prescription"},
]
payload_same = build_patient_payload(records_same)
check("REGRESSION CHECK: same patient_reference + hi_type still collapses into ONE patient object with 2 care contexts", len(payload_same) == 1 and len(payload_same[0]["careContexts"]) == 2)


PASS -- 2 distinct patient_reference values sharing an HI Type -> 2 separate patient objects (not merged/mislabeled)
PASS -- each patient object keeps its OWN patient_reference as referenceNumber
PASS -- each patient object has exactly its own 1 care context, not the other's
PASS -- REGRESSION CHECK: same patient_reference + hi_type still collapses into ONE patient object with 2 care contexts


True

## M2-34 / M3-7 — server launched from a different working directory

**Real bug found and fixed** (shared root cause for both cases). `server/callbacks/utils/storage.py`'s
`CALLBACK_FOLDER` and `server/callbacks/utils/api_capture.py`'s `CAPTURE_DIR` were both bare relative paths
(`Path("storage/callbacks")` / `Path("storage/api_capture")`), resolved against the process's **current
working directory at import time** — launching the server from anywhere other than the repo root silently
created/read a SECOND, disconnected `storage/` tree there. Fixed by anchoring both to `Path(__file__).resolve()`,
the same convention `json_file_store.py`'s `_STORAGE_ROOT` and `flow_logger.py`'s `_LOG_DIR` already use —
this is exactly the "M3 api-capture directory path is resolved relative to the launch directory" gap the
original test plan flagged for M3-7. Verified below by actually launching a subprocess from `/tmp` (not the
repo root) and confirming both paths still resolve to the real repo's `storage/` tree.

**M3-7's own honest caveat:** the test plan's M3-7 text also claims "the CLI's own polling code" resolves
its path *absolutely* (repo-root-anchored) while the api-capture side was relative — implying a CLI that
times out forever waiting for callbacks it can't see. This repo does not actually contain any `m2_test_suite`/
`m3_test_suite` CLI or a `wait_for_callback()` function (confirmed via a full repo-wide search) — only
`tools/m1_test_suite/` exists. So the SHARED root cause (the relative-path bug) is fixed and verified below,
but the specific "CLI times out" symptom described in the plan can't be reproduced or independently verified
here, since the CLI code it refers to isn't present in this codebase. Left as a caveat rather than claimed
Done outright — see the Notion Notes for this row.

In [5]:
# Simulate launching the server from a directory OTHER than the repo root
# by spawning a fresh Python process with cwd=/tmp (can't just os.chdir() in
# this same process -- storage.py/api_capture.py's module-level constants
# are already computed from THIS notebook process's own cwd at its own
# import time, so a real subprocess is needed to prove the fix holds for a
# genuinely fresh process launched elsewhere).
result = subprocess.run(
    [sys.executable, "-c",
     "import os, sys; os.chdir('/tmp'); sys.path.insert(0, os.environ['REPO_ROOT']); "
     "import server.callbacks.utils.storage as s; import server.callbacks.utils.api_capture as a; "
     "print(str(s.CALLBACK_FOLDER)); print(str(a.CAPTURE_DIR))"],
    capture_output=True, text=True, cwd="/tmp",
    env={**__import__('os').environ, "REPO_ROOT": str(__import__('pathlib').Path('.').resolve())},
)
lines = [l for l in result.stdout.strip().splitlines() if l]
repo_root = str(__import__('pathlib').Path('.').resolve())
check("launching from /tmp (not the repo root) -- CALLBACK_FOLDER still resolves under the real repo's storage/", len(lines) == 2 and lines[0] == f"{repo_root}/storage/callbacks")
check("launching from /tmp (not the repo root) -- CAPTURE_DIR still resolves under the real repo's storage/", len(lines) == 2 and lines[1] == f"{repo_root}/storage/api_capture")
if result.returncode != 0:
    print("subprocess stderr:", result.stderr)


NotADirectoryError: [WinError 267] The directory name is invalid

## M2-54 — link_confirm on-confirm ack lost forever on retry exhaustion

**Real bug found and fixed.** `link_confirm_service.py`'s `process_link_confirm()` ran `mark_processed()`
before `send_on_confirm()` (the ack). If `call_with_retry()` genuinely exhausted its retries and raised, the
outer try/except caught it and logged — but any LATER redelivery of the identical confirm request (same
REQUEST-ID, same `confirmation.linkRefNumber` — the CM/gateway forwards the same message, it's not something
we generate) hit the `already_processed()` branch, which just logged "already processed" and returned. The
ack was never re-attempted, ever. Fixed: the replay branch now re-derives the patient payload (session
lookup, patient identity lookup, record search — all read-only, safe to re-run) and re-sends `send_on_confirm()`,
WITHOUT re-verifying the OTP (a single-use token must not be re-consumed).

In [6]:
import server.callbacks.services.link_confirm_service as link_confirm_service
from server.callbacks.repository.link_repository import save_link_session
from server.callbacks.repository.patient_identity_repository import save_patient_identity

save_link_session("linkref-m254", {
    "otp_txn_id": "txn-otp-1",
    "abha_address": "m254@sbx",
    "selected_patient_records": [],
    "otp_expiry": None,
})
save_patient_identity("m254@sbx", {"abha_number": "12-3456-7890-1234", "hip_id": "IN3310002215"})

ack_recorder = CallRecorder(FakeResponse(202))
callback_data = {
    "headers": {"request-id": "req-m254-a"},
    "body": {"confirmation": {"token": "111111", "linkRefNumber": "linkref-m254"}},
}

with patch.object(link_confirm_service, "search_patient", lambda **kw: []), \
     patch.object(link_confirm_service, "send_on_confirm", ack_recorder), \
     patch.object(link_confirm_service, "verify_otp", lambda **kw: True):
    await link_confirm_service.process_link_confirm(callback_data)
    check("first delivery sends the ack once", ack_recorder.call_count == 1)

    # Simulate ABDM/the CM redelivering the identical confirm request --
    # e.g. because the first ack never actually reached them.
    await link_confirm_service.process_link_confirm(callback_data)
    check("FIX: replayed request re-sends the ack instead of silently no-op'ing forever", ack_recorder.call_count == 2)


2026-08-15 20:33:09  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 20:33:09     [API] Confirming Successful Link with ABDM -- POST .../on-confirm -> 202
2026-08-15 20:33:09  -> Patient's records are now linked
2026-08-15 20:33:09  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 20:33:09  -> REQUEST-ID req-m254-a already processed for link_confirm -- treating as a replay, skipping re-verification and re-confirmation, but re-sending the ack in case the first ack attempt never actually reached ABDM.
2026-08-15 20:33:09     [API] Confirming Successful Link with ABDM (replay -- re-sent ack) -- POST .../on-confirm -> 202


PASS -- first delivery sends the ack once
PASS -- FIX: replayed request re-sends the ack instead of silently no-op'ing forever


## M2-55 — link_init on-init ack lost forever on retry exhaustion

**Real bug found and fixed** (same shape as M2-54, but harder to recover from). `link_init_service.py`
generates `link_reference_number` itself and only tells ABDM about it via the on-init ack — so unlike M2-54,
a redelivered `link/init` request does NOT carry that value back to us; ABDM never received it if the first
ack failed. Fixed with a small reverse lookup: `_find_link_session_by_request_id()` scans stored link
sessions (via the existing, previously-unused `get_all_link_sessions()` debug helper) for the one whose own
`request_id` field matches the replay's REQUEST-ID, recovers `link_reference_number` from it, and re-sends
`send_on_init()` — without requesting a second OTP or saving a second session.

In [7]:
import server.callbacks.services.link_init_service as link_init_service
from server.callbacks.repository.patient_identity_repository import save_patient_identity as save_pid_55

save_pid_55("m255@sbx", {"abha_number": "12-3456-7890-5555"})

ack_recorder_55 = CallRecorder(FakeResponse(202))
otp_response = FakeResponse(200, {"txnId": "otp-txn-55"})
callback_data_55 = {
    "headers": {"request-id": "req-m255-a"},
    "body": {"abhaAddress": "m255@sbx", "transactionId": "txn-55", "patient": []},
}

otp_request_recorder = CallRecorder(otp_response)

with patch.object(link_init_service, "request_otp", otp_request_recorder), \
     patch.object(link_init_service, "encrypt_value", lambda *a, **kw: "encrypted"), \
     patch.object(link_init_service, "get_public_certificate", lambda: "cert"), \
     patch.object(link_init_service, "send_on_init", ack_recorder_55):
    await link_init_service.process_link_init(callback_data_55)
    check("first delivery sends the on-init ack once", ack_recorder_55.call_count == 1)
    check("first delivery requests exactly 1 OTP", otp_request_recorder.call_count == 1)

    # Simulate ABDM redelivering the identical link/init request.
    await link_init_service.process_link_init(callback_data_55)
    check("FIX: replayed request re-sends the on-init ack (link_reference_number recovered via reverse lookup)", ack_recorder_55.call_count == 2)
    check("FIX: replay does NOT request a second OTP", otp_request_recorder.call_count == 1)


2026-08-15 20:33:34  -> Patient chose to link records -- link request received (POST /api/v3/hip/link/care-context/init)
2026-08-15 20:33:34  -> Extracted care contexts the patient wants to link
2026-08-15 20:33:34     [API] Requesting OTP for Patient Verification -- POST .../profile/login/request/otp -> 200
2026-08-15 20:33:34     [API] Confirming Link Initiated with ABDM -- POST .../on-init -> 202
2026-08-15 20:33:34     [WAITING] Waiting for the patient to enter the OTP sent to their mobile/email
2026-08-15 20:33:34  -> Patient chose to link records -- link request received (POST /api/v3/hip/link/care-context/init)
2026-08-15 20:33:34  -> REQUEST-ID req-m255-a already processed for link_init -- treating as a replay, skipping a second OTP request/link session, but re-sending the on-init ack in case the first ack attempt never actually reached ABDM.
2026-08-15 20:33:34     [API] Confirming Link Initiated with ABDM (replay -- re-sent ack) -- POST .../on-init -> 202


PASS -- first delivery sends the on-init ack once
PASS -- first delivery requests exactly 1 OTP
PASS -- FIX: replayed request re-sends the on-init ack (link_reference_number recovered via reverse lookup)
PASS -- FIX: replay does NOT request a second OTP


## M2-56 — care context Notify Care Context Update: partial failure never retried

**Real bug found and fixed** (same root cause as M2-54/M2-55, different outbound call). The old code
unconditionally deleted the pending care-context-link record after its per-care-context notify loop, even
when SOME (not all) of those calls permanently failed after `call_with_retry`'s own retries exhausted — so
a genuinely failed notify had no recovery path, ever, even on a redelivery of the same `on_carecontext`
message. Fixed: the pending record is now only deleted once every care context notifies successfully;
a partial failure rewrites it down to just the still-failed subset, and a replay retries ONLY those, not
the ones that already succeeded.

In [8]:
import server.callbacks.services.care_context_link_service as ccl_service
from server.callbacks.repository.care_context_link_repository import save_pending_care_context_link, get_pending_care_context_link

save_pending_care_context_link("req-m256-a", {
    "hip_id": "IN3310002215",
    "abha_address": "m256@sbx",
    "link_token": "tok-m256",
    "patient_reference": "MR-256",
    "care_context_hi_types": {
        "CC-good": ["Prescription"],
        "CC-bad": ["Prescription"],
    },
})

call_log = []

def flaky_notify(**kwargs):
    call_log.append(kwargs["care_context_reference"])
    if kwargs["care_context_reference"] == "CC-bad":
        raise Exception("simulated permanent network failure (retries exhausted)")
    return FakeResponse(202)

callback_data_56 = {"body": {"abhaAddress": "m256@sbx", "status": "SUCCESS", "response": {"requestId": "req-m256-a"}}}

with patch.object(ccl_service, "notify_care_context_update", flaky_notify):
    await ccl_service.process_care_context_link(callback_data_56)

check("first delivery attempts both care contexts", set(call_log) == {"CC-good", "CC-bad"})
pending_after_first = get_pending_care_context_link("req-m256-a")
check("FIX: pending record kept (not deleted) since CC-bad permanently failed", pending_after_first is not None)
check("FIX: only the FAILED care context remains pending, the succeeded one is dropped from it", pending_after_first is not None and list(pending_after_first["care_context_hi_types"].keys()) == ["CC-bad"])

# Simulate ABDM redelivering the same on_carecontext message; this time the
# previously-flaky call succeeds.
call_log.clear()
with patch.object(ccl_service, "notify_care_context_update", lambda **kw: (call_log.append(kw["care_context_reference"]), FakeResponse(202))[1]):
    await ccl_service.process_care_context_link(callback_data_56)

check("FIX: replay retries ONLY the still-failed care context (CC-good not re-notified)", call_log == ["CC-bad"])
pending_after_second = get_pending_care_context_link("req-m256-a")
check("FIX: pending record finally deleted once every care context has succeeded", pending_after_second is None)


2026-08-15 20:33:49  -> ABDM confirmed the care context link result (POST /api/v3/link/on_carecontext)
2026-08-15 20:33:49  -> Care context successfully linked for m256@sbx: SUCCESS
2026-08-15 20:33:49     [API] Notifying care context update for CC-good -- POST .../link/context/notify -> 202
2026-08-15 20:33:49     [ERROR] Notify Care Context Update for CC-bad failed unexpectedly: simulated permanent network failure (retries exhausted) -- keeping it pending for a future retry rather than dropping it.
2026-08-15 20:33:49     [ERROR] 1 of 2 care context(s) still failed to notify for requestId 'req-m256-a' -- kept pending, not deleted, so a future redelivery of this on_carecontext message can retry them.
2026-08-15 20:33:49  -> ABDM confirmed the care context link result (POST /api/v3/link/on_carecontext)
2026-08-15 20:33:49  -> REQUEST-ID req-m256-a already processed for care_context_link, but 1 care context(s) are still pending from a prior partial failure -- retrying just those instead

PASS -- first delivery attempts both care contexts
PASS -- FIX: pending record kept (not deleted) since CC-bad permanently failed
PASS -- FIX: only the FAILED care context remains pending, the succeeded one is dropped from it
PASS -- FIX: replay retries ONLY the still-failed care context (CC-good not re-notified)
PASS -- FIX: pending record finally deleted once every care context has succeeded


True